<a href="https://colab.research.google.com/github/MuhammadRayan07/Weather-Analysis/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Data Analysis and Prediction
## Notebook 1: Data Collection

**Purpose:** Collect daily historical weather data for 20 cities (2016-2025) from the Open-Meteo Historical Weather API.
**Data source:** Open-Meteo (https://open-meteo.com), reanalysis-based data.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/weather-analysis")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Folders ready:", RAW.exists(), PROCESSED.exists())

Mounted at /content/drive
Folders ready: True True


In [2]:
import pandas as pd

cities = pd.DataFrame([
    ("Lahore","Pakistan",31.5497,74.3436),("Karachi","Pakistan",24.8607,67.0011),
    ("Dubai","UAE",25.2048,55.2708),("Delhi","India",28.6139,77.2090),
    ("Tokyo","Japan",35.6762,139.6503),("Singapore","Singapore",1.3521,103.8198),
    ("Beijing","China",39.9042,116.4074),("Sydney","Australia",-33.8688,151.2093),
    ("Cairo","Egypt",30.0444,31.2357),("Nairobi","Kenya",-1.2921,36.8219),
    ("Lagos","Nigeria",6.5244,3.3792),("Cape Town","South Africa",-33.9249,18.4241),
    ("London","UK",51.5074,-0.1278),("Paris","France",48.8566,2.3522),
    ("Moscow","Russia",55.7558,37.6173),("Reykjavik","Iceland",64.1466,-21.9426),
    ("New York","USA",40.7128,-74.0060),("Toronto","Canada",43.6532,-79.3832),
    ("Mexico City","Mexico",19.4326,-99.1332),("Sao Paulo","Brazil",-23.5505,-46.6333),
], columns=["city","country","latitude","longitude"])

cities.to_csv(PROCESSED / "city_reference.csv", index=False)
cities

,city,country,latitude,longitude
0,Lahore,Pakistan,31.5497,74.3436
1,Karachi,Pakistan,24.8607,67.0011
2,Dubai,UAE,25.2048,55.2708
3,Delhi,India,28.6139,77.2090
4,Tokyo,Japan,35.6762,139.6503
5,Singapore,Singapore,1.3521,103.8198
6,Beijing,China,39.9042,116.4074
7,Sydney,Australia,-33.8688,151.2093
8,Cairo,Egypt,30.0444,31.2357
9,Nairobi,Kenya,-1.2921,36.8219


Choosing cities with different weather conditions (a mix of tropical, desert, temperate, and polar climates across six continents)

In [3]:
import requests

URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY = ["weather_code","temperature_2m_max","temperature_2m_min","temperature_2m_mean",
         "precipitation_sum","rain_sum","precipitation_hours",
         "wind_speed_10m_max","wind_gusts_10m_max","wind_direction_10m_dominant",
         "shortwave_radiation_sum","sunshine_duration","daylight_duration"]

test = cities.iloc[0]
params = {"latitude": test.latitude, "longitude": test.longitude,
          "start_date": "2016-01-01", "end_date": "2025-12-31",
          "daily": ",".join(DAILY), "timezone": "auto"}

r = requests.get(URL , params=params ,timeout=60)
print("Status:", r.status_code)
data = r.json()
if r.status_code != 200:
    print(data)
else:
    print(data.keys())
    print(data["daily_units"])
    print(len(data["daily"]["time"]), "days")

Status: 200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': 'iso8601', 'weather_code': 'wmo code', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'precipitation_hours': 'h', 'wind_speed_10m_max': 'km/h', 'wind_gusts_10m_max': 'km/h', 'wind_direction_10m_dominant': '°', 'shortwave_radiation_sum': 'MJ/m²', 'sunshine_duration': 's', 'daylight_duration': 's'}
3653 days
